In [1]:
# Mounting to google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Uniziping
!unzip /content/drive/MyDrive/dataset.zip

In [30]:
# Creating Train and test dataset
import tensorflow as tf
img_size = (180, 180)
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    "dataset/training_set",
    image_size=img_size,
    batch_size=batch_size
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    "dataset/test_set",
    image_size=img_size,
    batch_size=batch_size
)

Found 8000 files belonging to 2 classes.
Found 2000 files belonging to 2 classes.


In [31]:
# Data augmentation
from tensorflow.keras import layers
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

In [33]:
# Defining our cnn model
from tensorflow.keras import layers

model = tf.keras.Sequential([
    layers.Input(shape=(180, 180, 3)),
    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(filters=16, kernel_size=3, padding='same', activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(filters=32, kernel_size=3, padding='same', activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(filters=64, kernel_size=3, padding='same', activation="relu"),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

In [34]:
with tf.device('/device:GPU:0'):
  model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
  model.fit(train_ds, validation_data=test_ds, epochs=10)

Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 15s 51ms/step - accuracy: 0.5500 - loss: 0.7162 - val_accuracy: 0.6345 - val_loss: 0.6370
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 19s 49ms/step - accuracy: 0.6391 - loss: 0.6363 - val_accuracy: 0.6685 - val_loss: 0.6202
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 11s 45ms/step - accuracy: 0.6891 - loss: 0.5901 - val_accuracy: 0.7150 - val_loss: 0.5576
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 12s 48ms/step - accuracy: 0.7159 - loss: 0.5538 - val_accuracy: 0.7270 - val_loss: 0.5433
Epoch 5/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 12s 48ms/step - accuracy: 0.7258 - loss: 0.5429 - val_accuracy: 0.7380 - val_loss: 0.5241
Epoch 6/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 12s 47ms/step - accuracy: 0.7518 - loss: 0.5070 - val_accuracy: 0.7675 - val_loss: 0.4966
Epoch 7/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 12s 48ms/step - accuracy: 0.7611 - loss: 0.4941 - val_accuracy: 0.7705 - val_loss: 0.4851
Epoch 8/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 12s 48ms/step - accuracy: 0.7768 - loss: 0.4799 - 

In [47]:
# Prediction function
import numpy as np
def predict_img(img_path):
  img = tf.keras.utils.load_img(img_path, target_size = (180, 180)) # Load image
  img_array = tf.keras.utils.img_to_array(img) # convert img to array
  img_array = np.expand_dims(img_array, axis=0) # change shape to (1, 180, 180, 3) -> since model expects a batch even for one image
  prediction = model.predict(img_array) # prediction will look like this [[0.83]]
  class_names = train_ds.class_names # class names will look like this ['cats', 'dogs']

  # predict image
  if prediction[0][0]>0.5:
    print("Predicted:", class_names[1])
  else:
    print("Predicted:", class_names[0])


In [49]:
predict_img("/content/cat_or_dog_1.jpg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
Predicted: dogs


In [48]:
predict_img("/content/cat_or_dog_2.jpg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
Predicted: cats
